# BZ running-experiment TPOT vs HQ TPOT

Compare **time-per-output-token (TPOT)** for last night's BZ runs against HQ dashboard snapshots
(`experiments_hq`).

| Series | Exp | Source of TPOT | Notes |
|---|---|---|---|
| BZ soak abort | **110** | `logs.json` `tpot_avg_s` | hf-lmsys us+boom, short (~273 reqs) |
| BZ overnight | **111** | `router_logs.json` `tpot_avg_s` | Claude/CodeFlow 96 users (still running) |
| HQ | `experiments_hq/1` | `identifier_metrics[*].tpot` | ms/token from zhongruan 24h API |

Units: notebook converts everything to **ms/token**. HQ values are already ms; BZ logs are seconds.


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BZ_ROOT = Path("/data/experiments")
HQ_ROOT = Path("/home/saeid/experiments_hq/1")

# Running / overnight BZ experiments
BZ_EXPS = {
    110: {"label": "BZ-110 soak (hf-lmsys, aborted)", "color": "#2ca02c"},
    111: {"label": "BZ-111 overnight (claude/codeflow, live)", "color": "#1f77b4"},
}

# HQ models to overlay (edit freely)
HQ_MODELS = [
    "MiniMax-M2.7-direct142",          # 142 reference
    "MiniMax-M2.7-llmla1-direct65",    # HQ us-path analog
    "MiniMax-M2.7-llmla2-direct65",
]

# Rolling window for BZ per-request TPOT → smooth series
ROLL_MIN = "5min"
print("BZ_ROOT", BZ_ROOT, "exists", BZ_ROOT.exists())
print("HQ_ROOT", HQ_ROOT, "exists", HQ_ROOT.exists(), "n_snapshots",
      len(list(HQ_ROOT.glob("*data_24hour*.json"))))


## 1. Load BZ TPOT over time

- Exp **110**: Boom OpenAI-compat path writes `tpot_avg_s` on `logs.json`.
- Exp **111**: Claude CLI `logs.json` has no TTFT/TPOT; use **`router_logs.json`** (router saw the upstream stream).


In [ ]:
def _read_ndjson(path: Path):
    rows = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_bz_tpot(exp_id: int, root: Path = BZ_ROOT) -> pd.DataFrame:
    """Return DataFrame with columns: ts (UTC), tpot_ms, endpoint_id, source."""
    d = root / str(exp_id)
    logs = d / "logs.json"
    router = d / "router_logs.json"

    # Prefer logs.json if it actually has tpot_avg_s; else router_logs.
    src_path, source = logs, "logs.json"
    if logs.is_file():
        probe = _read_ndjson(logs)
        n_tpot = sum(1 for r in probe if r.get("tpot_avg_s") is not None)
        if n_tpot == 0 and router.is_file():
            src_path, source = router, "router_logs.json"
            probe = _read_ndjson(router)
    elif router.is_file():
        src_path, source = router, "router_logs.json"
        probe = _read_ndjson(router)
    else:
        raise FileNotFoundError(f"no logs for exp {exp_id} in {d}")

    rows = []
    for r in probe:
        tpot = r.get("tpot_avg_s")
        if tpot is None:
            continue
        ts = r.get("t1_wall") or r.get("t0_wall") or r.get("actual_send_ts_wall")
        if ts is None:
            continue
        rows.append({
            "exp": exp_id,
            "ts": pd.to_datetime(ts, unit="s", utc=True),
            "tpot_ms": float(tpot) * 1000.0,
            "ttft_s": r.get("ttft_s"),
            "endpoint_id": r.get("endpoint_id"),
            "completion_tokens": r.get("completion_tokens"),
            "source": source,
        })
    df = pd.DataFrame(rows).sort_values("ts").reset_index(drop=True)
    print(f"exp {exp_id}: {len(df)} tpot samples from {source}"
          f" | med={df['tpot_ms'].median():.1f} ms"
          f" | p95={df['tpot_ms'].quantile(0.95):.1f} ms"
          f" | range {df['ts'].min()} → {df['ts'].max()}")
    return df


bz = {eid: load_bz_tpot(eid) for eid in BZ_EXPS}
bz_all = pd.concat(bz.values(), ignore_index=True) if bz else pd.DataFrame()
bz_all.head()


In [ ]:
def roll_median(df: pd.DataFrame, freq: str = ROLL_MIN) -> pd.DataFrame:
    if df.empty:
        return df
    s = (df.set_index("ts")["tpot_ms"]
           .resample(freq).median()
           .dropna()
           .rename("tpot_ms_med")
           .to_frame())
    s["tpot_ms_p95"] = df.set_index("ts")["tpot_ms"].resample(freq).quantile(0.95)
    s["n"] = df.set_index("ts")["tpot_ms"].resample(freq).count()
    return s.reset_index()


fig, ax = plt.subplots(figsize=(12, 4.5))
for eid, meta in BZ_EXPS.items():
    df = bz[eid]
    if df.empty:
        continue
    rolled = roll_median(df)
    ax.plot(rolled["ts"], rolled["tpot_ms_med"],
            color=meta["color"], lw=2, label=f'{meta["label"]} (median/{ROLL_MIN})')
    ax.fill_between(rolled["ts"], rolled["tpot_ms_med"], rolled["tpot_ms_p95"],
                    color=meta["color"], alpha=0.15, label=f'{eid} p95 band' if eid == 111 else None)

ax.set_ylabel("TPOT (ms / token)")
ax.set_xlabel("UTC")
ax.set_title("BZ running experiments — TPOT over time")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", fontsize=8)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# numeric summary
summary = []
for eid, df in bz.items():
    if df.empty:
        continue
    summary.append({
        "exp": eid,
        "n": len(df),
        "source": df["source"].iloc[0],
        "tpot_ms_p50": df["tpot_ms"].median(),
        "tpot_ms_p90": df["tpot_ms"].quantile(0.90),
        "tpot_ms_p95": df["tpot_ms"].quantile(0.95),
        "tpot_ms_max": df["tpot_ms"].max(),
        "start_utc": df["ts"].min(),
        "end_utc": df["ts"].max(),
    })
pd.DataFrame(summary)


## 2. Load HQ TPOT time series

HQ scrapes (`*api_data_24hour*.json`) store aligned `timestamps` + per-model `tpot` (already **ms/token**).
We take the **latest snapshot** that covers each model (edit `HQ_MODELS` above).


In [ ]:
def load_hq_tpot(hq_root: Path = HQ_ROOT, models=None) -> pd.DataFrame:
    models = models or HQ_MODELS
    snaps = sorted(hq_root.glob("*api_data_24hour*.json"))
    if not snaps:
        raise FileNotFoundError(f"no *api_data_24hour*.json under {hq_root}")

    # Prefer the newest snapshot; fall back if a model is missing.
    rows = []
    used = None
    for snap in reversed(snaps):
        d = json.load(snap.open())
        ts = d.get("timestamps") or []
        metrics = d.get("identifier_metrics") or {}
        if not any(m in metrics for m in models):
            continue
        used = snap
        for m in models:
            mdata = metrics.get(m) or {}
            tpot = mdata.get("tpot") or []
            for t, v in zip(ts, tpot):
                if v is None:
                    continue
                rows.append({
                    "model": m,
                    "ts": pd.to_datetime(t),  # local wall from HQ dashboard (no tz)
                    "tpot_ms": float(v),
                    "snapshot": snap.name,
                })
        break

    df = pd.DataFrame(rows)
    print("HQ snapshot:", used.name if used else None)
    if not df.empty:
        print(df.groupby("model")["tpot_ms"].agg(["count", "median", "min", "max"]))
        print("HQ time range:", df["ts"].min(), "→", df["ts"].max())
    return df


hq = load_hq_tpot()
hq.head()


In [ ]:
# HQ TPOT over its own wall-clock (last 24h window in the snapshot)
fig, ax = plt.subplots(figsize=(12, 4.5))
for m, g in hq.groupby("model"):
    g = g.sort_values("ts")
    ax.plot(g["ts"], g["tpot_ms"], lw=1.8, label=m)
ax.set_ylabel("TPOT (ms / token)")
ax.set_xlabel("HQ dashboard time")
ax.set_title("HQ TPOT over time (latest 24h snapshot)")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", fontsize=8)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 3. Side-by-side compare (BZ vs HQ)

Absolute clocks differ (BZ overnight vs HQ snapshot window). Two views:

1. **Absolute UTC / wall** — BZ vs HQ on their native clocks (shape / level comparison).
2. **Aligned from t=0** — each series shifted so its first sample is hour 0 (level + ramp comparison).


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharey=True)

# --- (1) absolute ---
ax = axes[0]
for eid, meta in BZ_EXPS.items():
    rolled = roll_median(bz[eid])
    if rolled.empty:
        continue
    ax.plot(rolled["ts"], rolled["tpot_ms_med"], color=meta["color"], lw=2,
            label=f"BZ {eid} med/{ROLL_MIN}")
for m, g in hq.groupby("model"):
    g = g.sort_values("ts")
    # HQ timestamps are naive; plot as-is for shape. Mark as HQ.
    ax.plot(g["ts"], g["tpot_ms"], lw=1.2, alpha=0.85, label=f"HQ {m}")
ax.set_title("Absolute time — BZ (UTC) vs HQ (dashboard local)")
ax.set_ylabel("TPOT (ms / token)")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", fontsize=7)

# --- (2) aligned from start ---
ax = axes[1]
for eid, meta in BZ_EXPS.items():
    rolled = roll_median(bz[eid])
    if rolled.empty:
        continue
    t0 = rolled["ts"].iloc[0]
    hours = (rolled["ts"] - t0).dt.total_seconds() / 3600.0
    ax.plot(hours, rolled["tpot_ms_med"], color=meta["color"], lw=2,
            label=f"BZ {eid}")
for m, g in hq.groupby("model"):
    g = g.sort_values("ts")
    t0 = g["ts"].iloc[0]
    hours = (g["ts"] - t0).dt.total_seconds() / 3600.0
    ax.plot(hours, g["tpot_ms"], lw=1.2, alpha=0.85, label=f"HQ {m}")
ax.set_xlabel("Hours since series start")
ax.set_ylabel("TPOT (ms / token)")
ax.set_title("Aligned from t=0 — level / ramp comparison")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", fontsize=7)

plt.tight_layout()
plt.show()

# level comparison table
rows = []
for eid, df in bz.items():
    if df.empty:
        continue
    rows.append({"series": f"BZ-{eid}", "p50_ms": df["tpot_ms"].median(),
                 "p95_ms": df["tpot_ms"].quantile(0.95), "n": len(df)})
for m, g in hq.groupby("model"):
    rows.append({"series": f"HQ-{m}", "p50_ms": g["tpot_ms"].median(),
                 "p95_ms": g["tpot_ms"].quantile(0.95), "n": len(g)})
pd.DataFrame(rows).sort_values("p50_ms")


## 4. Optional — restrict HQ window / add more BZ exps

After new sweeps finish, append ids to `BZ_EXPS` and re-run. To compare against a specific HQ day, set `HQ_SNAPSHOT` below to a filename stem and re-load.


In [ ]:
# Example: pin a specific HQ snapshot (None = newest with models present)
HQ_SNAPSHOT = None  # e.g. "20260714_114117_zhongruan_api_api_data_24hour.json"

def load_hq_tpot_pinned(name=None, models=None):
    models = models or HQ_MODELS
    if name is None:
        return load_hq_tpot(models=models)
    snap = HQ_ROOT / name
    d = json.load(snap.open())
    ts = d.get("timestamps") or []
    metrics = d.get("identifier_metrics") or {}
    rows = []
    for m in models:
        tpot = (metrics.get(m) or {}).get("tpot") or []
        for t, v in zip(ts, tpot):
            if v is None:
                continue
            rows.append({"model": m, "ts": pd.to_datetime(t), "tpot_ms": float(v),
                         "snapshot": snap.name})
    return pd.DataFrame(rows)

if HQ_SNAPSHOT:
    hq = load_hq_tpot_pinned(HQ_SNAPSHOT)
    print(hq.groupby("model")["tpot_ms"].median())
